# Rewards Value Realization & Incrementality Lab — Canonical Runbook

**Synthetic portfolio project only — NOT Starbucks actuals.**

This notebook is the single supported notebook workflow for the repository. It calls the canonical `src/` scripts rather than copying their code into notebook cells. It removes the earlier duplicated workflows and the hard-coded mock-results cell.

**Reference design:** seed `20261022`; 20,000 synthetic members; January–June 2026; April 1 randomized assignment; primary estimand is the treatment-minus-control difference in cumulative April–June contribution per randomized member.

**Current cloud status:** the uploaded notebook's saved execution record reports that BigQuery raw loads, modeling, and all 18 warehouse QA checks passed in `driiiportfolio.starbucks_rewards_lab_restart` on October 23, 2026. This notebook does not claim to independently verify the live cloud state until its optional read-only checks are run.


## 1. Repository setup

Run this notebook from a cloned repository. If the notebook opens in a clean Colab runtime, the next cell asks for the repository's HTTPS URL so it can clone the source. Do not paste credentials or access tokens.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path.cwd()
if not (REPO_DIR / "src" / "generate_data.py").exists():
    repo_url = input("Paste this portfolio repository's public HTTPS Git URL: ").strip()
    if not repo_url.startswith("https://github.com/") or not repo_url.endswith(".git"):
        raise ValueError("Enter the repository HTTPS URL ending in .git; do not include credentials.")
    REPO_DIR = Path("/content/starbucks-rewards-value-realization-lab")
    if REPO_DIR.exists():
        raise FileExistsError(f"{REPO_DIR} already exists but does not contain the expected source files. Inspect/remove it before cloning.")
    subprocess.run(["git", "clone", repo_url, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    os.chdir(REPO_DIR)

assert (REPO_DIR / "src" / "generate_data.py").exists(), "Repository source files not found."
print(f"Repository root: {REPO_DIR.resolve()}")


In [ ]:
%pip install -q -r requirements.txt


## 2. Regenerate, validate, analyze, and build deliverables

This sequence overwrites the synthetic CSVs and local output files with the deterministic reference run. It does **not** execute any BigQuery DDL, load jobs, or destructive cloud operations.


In [ ]:
def run_step(args):
    print("\n$", " ".join(map(str, args)))
    subprocess.run(list(map(str, args)), cwd=REPO_DIR, check=True)

run_step([sys.executable, "src/generate_data.py", "--members", "20000", "--seed", "20261022", "--out", "data/generated"])
run_step([sys.executable, "src/validate_data.py", "--data-dir", "data/generated"])
run_step([sys.executable, "-m", "pytest", "-q"])
run_step([sys.executable, "src/analyze_experiment.py", "--data-dir", "data/generated", "--out-dir", "outputs"])
run_step([sys.executable, "src/build_executive_artifacts.py", "--results", "outputs/experiment_results.json", "--out-dir", "outputs/executive_artifacts"])
print("\nLocal pipeline completed. Any failing step raises an exception and stops the sequence.")


In [ ]:
import json
from IPython.display import display
import pandas as pd

results_path = REPO_DIR / "outputs" / "experiment_results.json"
results = json.loads(results_path.read_text())
primary = results["primary_result"]
summary = pd.DataFrame([
    {"Metric": "Eligible randomized members", "Value": results["analysis_population_n"]},
    {"Metric": "Treatment N", "Value": primary["n_treatment"]},
    {"Metric": "Control N", "Value": primary["n_control"]},
    {"Metric": "Treatment mean contribution/member ($)", "Value": round(primary["mean_treatment"], 4)},
    {"Metric": "Control mean contribution/member ($)", "Value": round(primary["mean_control"], 4)},
    {"Metric": "ITT difference ($/member)", "Value": round(primary["difference_treatment_minus_control"], 4)},
    {"Metric": "95% CI lower ($/member)", "Value": round(primary["ci95_low"], 4)},
    {"Metric": "95% CI upper ($/member)", "Value": round(primary["ci95_high"], 4)},
    {"Metric": "Two-sided p-value", "Value": primary["p_value_two_sided"]},
    {"Metric": "Analysis run ID", "Value": results["analysis_run_id"]},
])
display(summary)

expected_files = [
    REPO_DIR / "outputs" / "experiment_results.json",
    REPO_DIR / "outputs" / "experiment_effects.csv",
    REPO_DIR / "outputs" / "looker_studio_results.csv",
    REPO_DIR / "outputs" / "member_level_analysis.csv",
    REPO_DIR / "outputs" / "executive_artifacts" / "Rewards_Experiment_Review.xlsx",
    REPO_DIR / "outputs" / "executive_artifacts" / "Rewards_Experiment_Executive_Readout.pptx",
]
for path in expected_files:
    assert path.exists() and path.stat().st_size > 0, f"Missing or empty deliverable: {path}"
    print(f"OK: {path.relative_to(REPO_DIR)} ({path.stat().st_size:,} bytes)")
assert results["synthetic_data_only"] is True


## 3. Optional read-only BigQuery verification

The cloud objects were already created according to the uploaded execution record. **Do not run the destructive reset in `sql/02_create_tables.sql` just to inspect them.** This section only reads table metadata/counts and the governed experiment view.

The Sandbox-specific fix uses `CLUSTER BY month_start, member_id` rather than time partitioning for member-month and benefit-event facts. This avoids the historical partition-purge issue but does not prevent Sandbox table/view objects themselves from expiring after about 60 days.

Run the next two cells only if you want to verify the current live project and reconcile SQL with the freshly generated Python results.


In [ ]:
# Colab authentication is interactive. This cell is read-only.
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
PROJECT_ID = "driiiportfolio"
DATASET_ID = "starbucks_rewards_lab_restart"
FQ = f"{PROJECT_ID}.{DATASET_ID}"
client = bigquery.Client(project=PROJECT_ID)

expected_rows = {
    "raw_members": 20000,
    "raw_member_month": 120000,
    "raw_benefit_events": 60000,
    "raw_market_context": 6,
    "dim_member": 20000,
    "fact_member_month": 120000,
    "fact_benefit_events": 60000,
    "dim_market_context": 6,
    "vw_experiment_member_outcomes": int(results["analysis_population_n"]),
}
cloud_check_rows = []
for table_name, expected in expected_rows.items():
    table = client.get_table(f"{FQ}.{table_name}")
    observed = int(client.query(f"SELECT COUNT(*) AS n FROM `{FQ}.{table_name}`").to_dataframe().iloc[0]["n"])
    cloud_check_rows.append({
        "table": table_name, "expected_rows": expected, "observed_rows": observed,
        "rows_match": observed == expected,
        "time_partitioned": table.time_partitioning is not None,
        "clustering_fields": ", ".join(table.clustering_fields or []),
        "expires": str(table.expires) if table.expires else "not reported",
    })
cloud_check = pd.DataFrame(cloud_check_rows)
display(cloud_check)
assert cloud_check["rows_match"].all(), "Cloud row-count mismatch: stop and investigate before dashboarding."
for table_name in ["raw_member_month", "raw_benefit_events", "fact_member_month", "fact_benefit_events"]:
    row = cloud_check.loc[cloud_check.table == table_name].iloc[0]
    assert not bool(row["time_partitioned"]), f"{table_name} is time-partitioned; check Sandbox expiration risk."
print("Read-only BigQuery row-count and partitioning checks passed.")


In [ ]:
# Reconcile Python arm means/counts with BigQuery's governed member-outcome view.
cloud_arm_sql = f"""
SELECT assignment_group, COUNT(*) AS n,
       AVG(post_contribution) AS mean_post_contribution,
       AVG(post_transactions) AS mean_post_transactions,
       AVG(post_net_sales) AS mean_post_net_sales
FROM `{FQ}.vw_experiment_member_outcomes`
GROUP BY assignment_group
"""
cloud_arms = client.query(cloud_arm_sql).to_dataframe().set_index("assignment_group")
python_outcome = results["primary_result"]
for group, n_key, mean_key in [
    ("TREATMENT", "n_treatment", "mean_treatment"),
    ("CONTROL", "n_control", "mean_control"),
]:
    expected_n = int(python_outcome[n_key])
    expected_mean = float(python_outcome[mean_key])
    actual_n = int(cloud_arms.loc[group, "n"])
    actual_mean = float(cloud_arms.loc[group, "mean_post_contribution"])
    assert actual_n == expected_n, f"{group} count mismatch: Python={expected_n}, BigQuery={actual_n}"
    assert abs(actual_mean - expected_mean) <= 0.011, (
        f"{group} contribution mean mismatch: Python={expected_mean:.6f}, BigQuery={actual_mean:.6f}"
    )
    print(f"{group}: N Python/BigQuery={expected_n}/{actual_n}; contribution mean Python/BigQuery=${expected_mean:.4f}/${actual_mean:.4f}")
print("Python/BigQuery arm counts and contribution means reconcile within $0.011.")


## 4. Interpretation guardrails

- The result is a randomized **intention-to-treat** comparison, not a comparison of benefit redeemers with non-redeemers.
- The planted 15% transaction-intensity multiplier and 80% exposure probability are synthetic design parameters. The ITT effect is estimated from outcomes and must not be presented as a Starbucks result.
- The contribution outcome already subtracts variable cost, ordinary reward cost, used-benefit cost, and intervention cost once. Do not subtract these costs again in scenario scaling.
- The purchaser-rate secondary outcome is not statistically distinguishable from zero at the 5% level in the current reference run. Secondary outcomes are exploratory.
- The dashboard remains a separate deliverable: build it from governed views only after the Python/BigQuery reconciliation cell passes.


## 6. Optional dashboard-source verification (run after cloud setup)

First run `sql/07_create_dashboard_views.sql`, upload the current `outputs/looker_studio_results.csv` as `driiiportfolio.starbucks_rewards_lab_restart.looker_studio_results` using `dashboard/looker_studio_results_schema.json` with overwrite semantics, and run `sql/08_dashboard_validation.sql`. Then run the following read-only check. This cell does not create or modify cloud objects.


In [ ]:
# Run only after the dashboard views and inference table have been created in BigQuery.
dashboard_objects = {
    "vw_dashboard_monthly_arm_performance": 12,
    "vw_dashboard_experiment_arm_summary": 2,
    "vw_dashboard_tier_arm_summary": 6,
    "vw_dashboard_benefit_arm_summary": 6,
    "looker_studio_results": 6,
}
dashboard_rows = []
for object_name, expected in dashboard_objects.items():
    observed = int(client.query(f"SELECT COUNT(*) AS n FROM `{FQ}.{object_name}`").to_dataframe().iloc[0]["n"])
    dashboard_rows.append({"object": object_name, "expected_rows": expected, "observed_rows": observed, "rows_match": observed == expected})
dashboard_check = pd.DataFrame(dashboard_rows)
display(dashboard_check)
assert dashboard_check["rows_match"].all(), "Dashboard source count mismatch; inspect SQL and the loaded inference CSV."
primary_cloud = client.query(f"SELECT analysis_run_id, effect_estimate, ci95_low, ci95_high FROM `{FQ}.looker_studio_results` WHERE result_type = 'ITT_OUTCOME' AND outcome = 'post_contribution'").to_dataframe()
assert len(primary_cloud) == 1, "Expected exactly one primary ITT result row."
assert primary_cloud.iloc[0]["analysis_run_id"] == results["analysis_run_id"], "Dashboard result table is stale relative to the local analysis run."
assert abs(float(primary_cloud.iloc[0]["effect_estimate"]) - float(results["primary_result"]["difference_treatment_minus_control"])) < 1e-9
print("Dashboard sources and primary result run ID reconciled.")
